In [ ]:
import pandas as pd
from openai import OpenAI
import numpy as np

df = pd.read_parquet("experiments/raw/Qwen_Qwen2.5-3B-Instruct.parquet")
df.head()

In [ ]:
client = OpenAI()
EMBED_MODEL = "text-embedding-3-small"

def embed(texts: list[str], batch_size: int = 2048) -> np.ndarray:
    """Return L2-normalized embeddings for a list of strings."""
    vecs = []
    for i in range(0, len(texts), batch_size):
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts[i : i + batch_size])
        vecs.extend(d.embedding for d in resp.data)
    arr = np.array(vecs, dtype=np.float32)
    arr /= np.linalg.norm(arr, axis=1, keepdims=True)
    return arr

In [ ]:
EMBED_MODEL = "text-embedding-3-small"
_gen_slug = df["model"].iloc[0].replace("/", "_")
_emb_slug = EMBED_MODEL.replace("/", "_")
_slug = f"{_gen_slug}__{_emb_slug}"

emb = np.load(f"results/embeddings/{_slug}_llm.npy")
llm_idx = pd.read_parquet(f"results/embeddings/{_slug}_llm_index.parquet")
# emb[i] corresponds to llm_idx.iloc[i] — rows sorted by (prompt_id, sample_idx)

In [ ]:
emb

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=0).fit(emb)
xy = pca.transform(emb)

fig, ax = plt.subplots(figsize=(9, 7))
langs = sorted(df.language.unique())
cmap = plt.get_cmap("tab20", len(langs))
for i, lang in enumerate(langs):
    m = (df.language == lang).to_numpy()
    ax.scatter(xy[m, 0], xy[m, 1], s=4, alpha=0.4, color=cmap(i), label=lang)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title(f"PCA of text-embedding-3-small embeddings — {df.model.iloc[0]}")
ax.legend(markerscale=3, ncols=2, fontsize=9, loc="best")
plt.tight_layout()
plt.show()

In [ ]:
human_emb = np.load(f"results/embeddings/{_slug}_human.npy")
human_idx = pd.read_parquet(f"results/embeddings/{_slug}_human_index.parquet")
# human_idx has columns: prompt_id, language, human_idx

# refit PCA on the union so both clouds share the same space
all_emb = np.vstack([emb, human_emb])
pca2 = PCA(n_components=2, random_state=0).fit(all_emb)
xy_m = pca2.transform(emb)
xy_h = pca2.transform(human_emb)

fig, ax = plt.subplots(figsize=(9, 7))
for i, lang in enumerate(langs):
    mm = (llm_idx.language == lang).to_numpy()
    hm = (human_idx.language == lang).to_numpy()
    ax.scatter(xy_m[mm, 0], xy_m[mm, 1], s=4, alpha=0.3, color=cmap(i), label=f"{lang} (LLM)")
    ax.scatter(xy_h[hm, 0], xy_h[hm, 1], s=40, marker="x", color=cmap(i),
               edgecolors="black", linewidths=0.8, label=f"{lang} (human)")
ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]:.1%})")
ax.set_title(f"{EMBED_MODEL}: LLM samples (dots) vs human responses (×) — {df.model.iloc[0]}")
ax.legend(markerscale=2, ncols=2, fontsize=7, loc="best")
plt.tight_layout()
plt.show()
print(f"LLM: {len(emb)} embeddings   Human: {len(human_emb)} embeddings")

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

langs = sorted(df.language.unique())
cmap = plt.get_cmap("tab20", len(langs))

ncols = 4
nrows = math.ceil(len(langs) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.8 * nrows))
axes = axes.flatten()

for i, lang in enumerate(langs):
    ax = axes[i]
    mm = (llm_idx.language == lang).to_numpy()
    hm = (human_idx.language == lang).to_numpy()

    lang_emb = np.vstack([emb[mm], human_emb[hm]])
    pca_l = PCA(n_components=2, random_state=0).fit(lang_emb)
    xy_lm = pca_l.transform(emb[mm])
    xy_lh = pca_l.transform(human_emb[hm])

    ax.scatter(xy_lm[:, 0], xy_lm[:, 1], s=6, alpha=0.35, color=cmap(i), label="LLM")
    ax.scatter(xy_lh[:, 0], xy_lh[:, 1], s=45, marker="x", color="black", linewidths=1.0, label="human")
    ax.set_title(f"{lang}  (n_LLM={mm.sum()}, n_human={hm.sum()})", fontsize=10)
    ax.set_xlabel(f"PC1 ({pca_l.explained_variance_ratio_[0]:.0%})", fontsize=8)
    ax.set_ylabel(f"PC2 ({pca_l.explained_variance_ratio_[1]:.0%})", fontsize=8)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8, loc="best")

for j in range(len(langs), len(axes)):
    axes[j].set_visible(False)

fig.suptitle(f"Per-language PCA: LLM samples vs human responses — {df.model.iloc[0]}", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Smoke set: 4 prompts (2 en + 2 fr) × 50 samples per model. Per-language PCA,
# LLM samples vs human responses, one row per model.

smoke_files = sorted(Path("experiments/raw_smoke").glob("*.parquet"))
models_dfs = [(p.stem, pd.read_parquet(p)) for p in smoke_files]
langs_sm = sorted({l for _, d in models_dfs for l in d.language.unique()})
cmap_sm = plt.get_cmap("tab20", max(len(langs_sm), 2))

ncols = len(langs_sm)
nrows = len(models_dfs)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.8 * nrows), squeeze=False)

emb_dir = Path("results/embeddings")

for r, (name, d) in enumerate(models_dfs):
    gen_model = d["model"].iloc[0]
    slug = f"{gen_model.replace('/', '_')}__{EMBED_MODEL.replace('/', '_')}"
    llm_emb = np.load(emb_dir / f"{slug}_llm.npy")
    llm_i = pd.read_parquet(emb_dir / f"{slug}_llm_index.parquet")
    h_emb = np.load(emb_dir / f"{slug}_human.npy")
    h_i = pd.read_parquet(emb_dir / f"{slug}_human_index.parquet")

    for c, lang in enumerate(langs_sm):
        ax = axes[r, c]
        mm = (llm_i.language == lang).to_numpy()
        hm = (h_i.language == lang).to_numpy()
        if mm.sum() == 0:
            ax.set_visible(False)
            continue
        lang_emb = np.vstack([llm_emb[mm], h_emb[hm]])
        pca_l = PCA(n_components=2, random_state=0).fit(lang_emb)
        xy_lm = pca_l.transform(llm_emb[mm])
        xy_lh = pca_l.transform(h_emb[hm])
        ax.scatter(xy_lm[:, 0], xy_lm[:, 1], s=6, alpha=0.4, color=cmap_sm(c), label="LLM")
        ax.scatter(xy_lh[:, 0], xy_lh[:, 1], s=45, marker="x", color="black", linewidths=1.0, label="human")
        ax.set_title(f"{name}  —  {lang}  (LLM={mm.sum()}, human={hm.sum()})", fontsize=9)
        ax.set_xlabel(f"PC1 ({pca_l.explained_variance_ratio_[0]:.0%})", fontsize=8)
        ax.set_ylabel(f"PC2 ({pca_l.explained_variance_ratio_[1]:.0%})", fontsize=8)
        ax.tick_params(labelsize=7)
        if r == 0 and c == 0:
            ax.legend(fontsize=8, loc="best")

fig.suptitle("Smoke run: per-language PCA per model", fontsize=12)
plt.tight_layout()
plt.show()

In [2]:
import pandas as pd
df = pd.read_parquet('results/all_generations.parquet')

In [ ]:
df

,model,dataset,language,prompt_id,prompt_text,human_responses,sample_idx,response_text,n_tokens,finish_reason
0,Qwen/Qwen2.5-3B-Instruct,oasst2_prompts,en,0348b67c-4388-4101-b50b-8cb1dd543edb,Give an opening strategy for the United Kingdo...,[Hearts of Iron 4 is a complex and challenging...,0,"In Hearts of Iron 4 (HOI4), crafting a success...",849,stop
1,Qwen/Qwen2.5-3B-Instruct,oasst2_prompts,en,0348b67c-4388-4101-b50b-8cb1dd543edb,Give an opening strategy for the United Kingdo...,[Hearts of Iron 4 is a complex and challenging...,1,"In Hearts of Iron 4 (HoI4), as the United King...",778,stop
2,Qwen/Qwen2.5-3B-Instruct,oasst2_prompts,en,0348b67c-4388-4101-b50b-8cb1dd543edb,Give an opening strategy for the United Kingdo...,[Hearts of Iron 4 is a complex and challenging...,2,"In Hearts of Iron 4 (HoI4), strategy involves ...",783,stop
3,Qwen/Qwen2.5-3B-Instruct,oasst2_prompts,en,0348b67c-4388-4101-b50b-8cb1dd543edb,Give an opening strategy for the United Kingdo...,[Hearts of Iron 4 is a complex and challenging...,3,"In Hearts of Iron 4, the opening strategy for ...",813,stop
4,Qwen/Qwen2.5-3B-Instruct,oasst2_prompts,en,0348b67c-4388-4101-b50b-8cb1dd543edb,Give an opening strategy for the United Kingdo...,[Hearts of Iron 4 is a complex and challenging...,4,"In ""Hearts of Iron IV,"" the United Kingdom is ...",788,stop
...,...,...,...,...,...,...,...,...,...,...
58495,meta-llama/Llama-3.2-3B-Instruct,oasst2_prompts,vi,f886e5e8-87f7-4448-8759-a04ee918ae6e,Viết dòng code Python để tương tác với API của...,[Để viết dòng code Python để tương tác với API...,45,Dưới đây là một ví dụ về cách tương tác với AP...,584,stop
58496,meta-llama/Llama-3.2-3B-Instruct,oasst2_prompts,vi,f886e5e8-87f7-4448-8759-a04ee918ae6e,Viết dòng code Python để tương tác với API của...,[Để viết dòng code Python để tương tác với API...,46,"Để tương tác với API của Google, bạn cần sử dụ...",879,stop
58497,meta-llama/Llama-3.2-3B-Instruct,oasst2_prompts,vi,f886e5e8-87f7-4448-8759-a04ee918ae6e,Viết dòng code Python để tương tác với API của...,[Để viết dòng code Python để tương tác với API...,47,"Để tương tác với API của Google, bạn cần sử dụ...",327,stop
58498,meta-llama/Llama-3.2-3B-Instruct,oasst2_prompts,vi,f886e5e8-87f7-4448-8759-a04ee918ae6e,Viết dòng code Python để tương tác với API của...,[Để viết dòng code Python để tương tác với API...,48,"Để tương tác với API của Google, bạn có thể sử...",634,stop
